In [ ]:
# Install
!pip install pandas scikit-learn nltk matplotlib -q

# Imports
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

nltk.download("stopwords")

# Load dataset
df = pd.read_csv("hobbs-pearson-trials.csv")

print("Shape:", df.shape)
print(df.head())
print(df.columns)

# Select text column automatically
text_col = df.select_dtypes(include="object").columns[0]

# NLP cleaning
stop = set(stopwords.words("english"))

def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return " ".join(
        word for word in text.split()
        if word not in stop and len(word) > 2
    )

df["clean_text"] = df[text_col].fillna("").apply(clean)

# TF-IDF
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df["clean_text"])

# K-Means
k = 4
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = model.fit_predict(X)

# Show clusters
print("\nCluster counts:")
print(df["cluster"].value_counts().sort_index())

# Important words in each cluster
words = tfidf.get_feature_names_out()

for i in range(k):
    top = model.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:")
    print(", ".join(words[j] for j in top))

# PCA visualisation
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X.toarray())

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["cluster"], cmap="viridis")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means Clustering of Hobbs-Pearson Trials")
plt.colorbar(label="Cluster")
plt.show()

# Save results
df.to_csv("hobbs-pearson-trials-clusters.csv", index=False)

print("\nDone! Results saved.")